# Copper V16-11 — SA-LSTM & SA-GRU optimization (Standalone)

File chạy **độc lập** với notebook còn lại; không cần đọc checkpoint từ notebook khác. `FULL_MODE` ở đây là FULL trong **ngân sách huấn luyện an toàn hơn**, KHÔNG có nghĩa tái hiện toàn bộ 650 scenarios của paper. Không thể cam kết chắc chắn thời gian GPU nếu Kaggle thay đổi quota/tốc độ. Dữ liệu từ `dataset_9_commodities_daily.csv`.

**Mặc định HORIZON=0:** X[t-21:t] (bao gồm WTI/Gold/Silver ngày t) → Copper(t) (nowcasting). `HORIZON=1` mới là dự báo trước khi có thông tin ngày t. **Residual models** thêm Copper(t−1) để tính ΔCopper và phải so sánh riêng với Paper3.


## 1. Cấu hình và môi trường


In [ ]:
from pathlib import Path
import zipfile
from datetime import datetime, timezone
import os, gc, json, time, math, hashlib, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
import tensorflow as tf


SEED=42
MODE='FULL_MODE'  # QUICK_MODE | FULL_MODE
MODE_CFG={
 'QUICK_MODE':{'epochs':4,'search_epochs':3,'sa_evals':5,'earlystop':True},
 'FULL_MODE':{'epochs':100,'search_epochs':12,'sa_evals':35,'earlystop':True},
}
CFG=MODE_CFG[MODE]
LOOKBACK=22
HORIZON=0
RATIOS=[.90,.925,.95]
BATCH_SIZE=64
FEATURE_MODES=['paper3']
RUN_ADAPTIVE=False
RUN_SA_GRU=True
MAX_RUNTIME_MINUTES=180  # Dừng SA trước ngưỡng, lưu tiến độ (không đảm bảo session không ngắt)
JOB_START=time.monotonic()
DATASET_CANDIDATES=[
 Path('/kaggle/input/datasets/lancelot71000/dataset-9-commodities-daily/dataset_9_commodities_daily.csv'),
 Path('/kaggle/input/dataset-9-commodities-daily/dataset_9_commodities_daily.csv'),
 Path('data/processed/dataset_9_commodities_daily.csv'),
]
OUT=Path('/kaggle/working/copper_sa_v16') if Path('/kaggle/working').exists() else Path('copper_sa_v16')
OUT.mkdir(parents=True,exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices('GPU'):
 try: tf.config.experimental.set_memory_growth(gpu,True)
 except RuntimeError: pass
print('Standalone SA notebook', MODE, 'GPU:', tf.config.list_physical_devices('GPU'))


## 2. Dataset
NOTEBOOK_NAME='SA models'


In [ ]:
TARGET = 'Copper'
PAPER3 = ['WTI', 'Gold', 'Silver']
LEGACY14 = ['Aluminium', 'Tin', 'Silver', 'Gold']
CANDIDATES = ['Aluminium', 'Nickel', 'Tin', 'Iron_Ore_62', 'Silver', 'Gold', 'WTI', 'Brent']

DATA_PATH = next((p for p in DATASET_CANDIDATES if p.is_file()), None)
if DATA_PATH is None and Path('/kaggle/input').exists():
    matches = list(Path('/kaggle/input').rglob('dataset_9_commodities_daily.csv'))
    if len(matches) == 1: DATA_PATH = matches[0]
if DATA_PATH is None: raise FileNotFoundError('Không tìm thấy dataset_9_commodities_daily.csv; gắn đúng Kaggle dataset.')
raw = pd.read_csv(DATA_PATH)
if 'Date' not in raw.columns or TARGET not in raw.columns:
    raise KeyError('CSV phải có Date và Copper.')
raw['Date'] = pd.to_datetime(raw['Date'], errors='coerce')
for name in [TARGET]+CANDIDATES:
    if name in raw: raw[name] = pd.to_numeric(raw[name], errors='coerce')
if raw['Date'].isna().any(): raise ValueError('Date có dòng không hợp lệ.')
if raw['Date'].duplicated().any(): raise ValueError('Có ngày trùng lặp; xử lý nguồn dữ liệu trước.')
missing = sorted(set(PAPER3+LEGACY14)-set(raw.columns))
if missing: raise ValueError(f'Thiếu cột yêu cầu: {missing}')
available_candidates = [c for c in CANDIDATES if c in raw]
raw = raw.sort_values('Date').reset_index(drop=True)
quality = pd.DataFrame({'column':raw.columns,'missing':raw.isna().sum().values,'dtype':raw.dtypes.astype(str).values})
quality.to_csv(OUT/'data_quality.csv', index=False)
# Để so sánh các nhánh trên CÙNG ngày, loại những ngày thiếu bất kỳ biến ứng viên hiện có.
# Điều này là quy tắc làm sạch chủ động được ghi nhận, không phải chắc chắn như paper.
data = raw.dropna(subset=[TARGET]+available_candidates).reset_index(drop=True)
assert len(data)>1000 and data['Date'].is_monotonic_increasing
print('Dataset:',DATA_PATH, '| raw:',len(raw),'| clean:',len(data), '|',data.Date.iloc[0].date(),'->',data.Date.iloc[-1].date())
display(quality)


## 3. Tách chuỗi không leakage


In [ ]:
def choose_features(frame, fit_start, val_start, feature_mode):
    train = frame.iloc[fit_start:val_start]
    if feature_mode == 'paper3':
        features = PAPER3.copy()
    elif feature_mode == 'legacy14':
        features = LEGACY14.copy()  # fixed a-priori; only for reproducing 14-11 setting
    elif feature_mode == 'train_top4':
        score = train[[TARGET]+available_candidates].corr(method='spearman')[TARGET].drop(TARGET)
        features = score.abs().sort_values(ascending=False).head(4).index.tolist()
    else: raise ValueError(feature_mode)
    corr = train[[TARGET]+available_candidates].corr(method='spearman')[TARGET].drop(TARGET)
    return features, corr

def prepare_context(frame, test_ratio, scheme='expanding', feature_mode='paper3', val_fraction=.10):
    n = len(frame)
    test_start = int(n*test_ratio)
    val_start = int(test_start*(1-val_fraction))
    assert LOOKBACK+HORIZON < val_start < test_start < n
    if scheme=='expanding' or scheme=='recency_weighted': fit_start=0
    elif scheme=='rolling_756': fit_start=max(0, val_start-756)
    elif scheme=='rolling_1260': fit_start=max(0, val_start-1260)
    else: raise ValueError(scheme)
    features, corr = choose_features(frame, fit_start, val_start, feature_mode)
    xsc = MinMaxScaler().fit(frame.iloc[fit_start:val_start][features])
    ysc = MinMaxScaler().fit(frame.iloc[fit_start:val_start][[TARGET]])
    x = xsc.transform(frame[features]).astype('float32')
    y = ysc.transform(frame[[TARGET]]).ravel().astype('float32')
    prev = np.r_[np.nan, frame[TARGET].to_numpy()[:-1]]
    prev_scaled = (prev - float(ysc.data_min_[0])) / float(ysc.data_range_[0])

    def windows(start,end, strictly_inside_train=False):
        idx = np.arange(start,end)
        idx = idx[(idx-HORIZON-LOOKBACK+1)>=0]
        if strictly_inside_train:
            idx = idx[(idx-HORIZON-LOOKBACK+1)>=fit_start]
        XX = np.stack([x[t-HORIZON-LOOKBACK+1:t-HORIZON+1] for t in idx]).astype('float32')
        YY = y[idx]
        return {'X':XX,'y':YY,'idx':idx,'prev':prev_scaled[idx].astype('float32'),
                'dates':frame.Date.iloc[idx].to_numpy()}
    train=windows(fit_start,val_start,True)
    val=windows(val_start,test_start)
    test=windows(test_start,n)
    assert max(train['idx'])<min(val['idx'])<min(test['idx'])
    assert np.max(train['idx']-HORIZON) < test_start
    assert xsc.n_samples_seen_ == val_start-fit_start
    return {'train':train,'val':val,'test':test,'x_scaler':xsc,'y_scaler':ysc,
            'features':features,'corr':corr,'fit_start':fit_start,'val_start':val_start,
            'test_start':test_start,'scheme':scheme,'ratio':test_ratio,'feature_mode':feature_mode}

# Independent smoke assertions for alignment, no test labels in input, scaler fit on train only.
ctx0=prepare_context(data,0.90)
assert ctx0['train']['X'].shape[1:] == (LOOKBACK,3)
assert len(set(ctx0['train']['idx']).intersection(ctx0['test']['idx']))==0
for ratio in RATIOS:
    c=prepare_context(data,ratio)
    print(f'{ratio:.3f}: train={len(c["train"]["idx"])} val={len(c["val"]["idx"])} test={len(c["test"]["idx"])} | {str(data.Date.iloc[c["test_start"]].date())}')


## 4. Mô hình


In [ ]:
PAPER_SA_LSTM_HP = {'units_1': 39, 'units_2': 111, 'learning_rate': 5e-5}

def build_paper_sa_lstm(input_shape, hp=None):
    # Hai lớp LSTM và bộ 39/111, LR=5e-5 được paper công bố.
    # Dense ReLU, output linear và Adam là các giả định triển khai đã được ghi nhận.
    model = tf.keras.Sequential(name='Paper_SA_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(39,
                                    return_sequences=True , 
                                    activation='tanh',
                                    recurrent_activation='sigmoid'))
    
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(111,
                                   activation='tanh',
                                    recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-5), loss='mse')
    return model


In [ ]:
def build_reoptimized_sa_lstm(input_shape, hp):
    model = tf.keras.Sequential(name='Reoptimized_SA_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(int(hp['units_1']), return_sequences=True,
                                activation='tanh',
                                recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(int(hp['units_2']), 
                                activation='tanh',
                                recurrent_activation='sigmoid'  ))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    optimizer = tf.keras.optimizers.Adam(float(hp['learning_rate']))
    model.compile(optimizer=optimizer, loss='mse')
    return model


In [ ]:
def build_sa_gru(input_shape, hp):
    model = tf.keras.Sequential(name='SA_GRU')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.GRU(int(hp['units_1']), return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.GRU(int(hp['units_2'])))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    optimizer = tf.keras.optimizers.Adam(float(hp['learning_rate']))
    model.compile(optimizer=optimizer, loss='mse')
    return model


In [ ]:
def build_matched_sa_lstm(input_shape,hp):
    model=tf.keras.Sequential(name='Matched_SA_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(int(hp['units_1']),return_sequences=True))
    model.add(tf.keras.layers.Dropout(.20))
    model.add(tf.keras.layers.LSTM(int(hp['units_2'])))
    model.add(tf.keras.layers.Dense(32,activation='relu'))
    model.add(tf.keras.layers.Dropout(.20))
    model.add(tf.keras.layers.Dense(1,activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(float(hp['learning_rate'])),loss='mse')
    return model


## 5. Huấn luyện và metrics


In [ ]:
def reset_keras(seed=SEED):
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(int(seed))

def weights_for_context(ctx):
    n=len(ctx['train']['idx']); age=np.arange(n-1,-1,-1,dtype=float)
    w=np.exp(-np.log(2)*age/252)
    return (w/np.mean(w)).astype('float32')

def fit_predict(builder,ctx,hp=None,seed=SEED,epochs=None,scheme=None,residual=False, search=False, skip_test=False):
    reset_keras(seed)
    tr,va = ctx['train'],ctx['val']
    te = None if skip_test else ctx['test']
    model = builder(tr['X'].shape[1:],hp)
    callbacks=[]
    if CFG['earlystop']:
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=7 if not search else 4,restore_best_weights=True)]
    ytr=tr['y']-tr['prev'] if residual else tr['y']
    yva=va['y']-va['prev'] if residual else va['y']
    weights=weights_for_context(ctx) if scheme=='recency_weighted' else None
    begin=time.perf_counter()
    hist=model.fit(tr['X'],ytr,validation_data=(va['X'],yva),epochs=epochs or CFG['epochs'],
                   batch_size=BATCH_SIZE,shuffle=False,callbacks=callbacks,verbose=0,sample_weight=weights)
    seconds=time.perf_counter()-begin
    pv=model.predict(va['X'],verbose=0).ravel()
    pt=model.predict(te['X'],verbose=0).ravel() if te is not None else None
    if residual:
        pv=pv+va['prev']
        if te is not None: pt=pt+te['prev']
    val_mse=float(mean_squared_error(va['y'],pv))
    ret={'val_pred':pv,'test_pred':pt,'val_mse':val_mse,'seconds':seconds,'epochs_run':len(hist.history['loss']),
         'loss_history':hist.history}
    del model;reset_keras(seed)
    return ret

def ridge_residual_prediction(ctx):
    tr,va,te=ctx['train'],ctx['val'],ctx['test']
    model=Ridge(alpha=1.)
    model.fit(tr['X'].reshape(len(tr['X']),-1),tr['y']-tr['prev'])
    pv=va['prev']+model.predict(va['X'].reshape(len(va['X']),-1))
    pt=te['prev']+model.predict(te['X'].reshape(len(te['X']),-1))
    return {'val_pred':pv,'test_pred':pt,'val_mse':float(mean_squared_error(va['y'],pv)),
            'seconds':0.,'epochs_run':0,'loss_history':None}

def evaluate_predictions(ctx,pred,model,seed,branch,scheme):
    true=ctx['test']['y'];ysc=ctx['y_scaler']
    true_raw=ysc.inverse_transform(true.reshape(-1,1)).ravel()
    pred_raw=ysc.inverse_transform(np.asarray(pred['test_pred']).reshape(-1,1)).ravel()
    persistence_scaled=ctx['test']['prev']
    naive_mae=float(mean_absolute_error(true,persistence_scaled))
    row={'branch':branch,'feature_mode':ctx['feature_mode'],'scheme':scheme,'ratio':ctx['ratio'],
         'horizon':HORIZON,'model':model,'seed':seed,'features':','.join(ctx['features']),
         'n_test':len(true),'train_start':str(data.Date.iloc[ctx['fit_start']].date()),
         'test_start':str(data.Date.iloc[ctx['test_start']].date()),
         'val_mse_scaled':pred['val_mse'],'MSE_scaled':float(mean_squared_error(true,pred['test_pred'])),
         'RMSE_raw':float(mean_squared_error(true_raw,pred_raw)**.5),
         'MAE_raw':float(mean_absolute_error(true_raw,pred_raw)),
         'R2_raw':float(r2_score(true_raw,pred_raw)),
         'Relative_MAE_vs_test_persistence':float(mean_absolute_error(true,pred['test_pred'])/naive_mae) if naive_mae>0 else np.nan,
         'MASE':float(mean_absolute_error(true_raw,pred_raw)/np.mean(np.abs(np.diff(data[TARGET].iloc[ctx['fit_start']:ctx['val_start']].to_numpy())))),
         'epochs_run':pred['epochs_run'],'train_seconds':pred['seconds']}
    preds=pd.DataFrame({'Date':ctx['test']['dates'],'ratio':ctx['ratio'],'branch':branch,'scheme':scheme,'feature_mode':ctx['feature_mode'],
                         'model':model,'seed':seed,'actual_raw':true_raw,'pred_raw':pred_raw})
    return row,preds

# Resumption within persistent output directory; no notebook cross-dependency.
result_file=OUT/'results_v16.csv'; pred_file=OUT/'predictions_v16.csv'
RESULT_ROWS = pd.read_csv(result_file).to_dict('records') if result_file.exists() else []
PREDICTIONS = [pd.read_csv(pred_file)] if pred_file.exists() else []

def has_result(ctx,model,seed,branch,scheme):
    return any(str(r.get('model'))==str(model) and str(r.get('branch'))==str(branch)
               and str(r.get('scheme'))==str(scheme)
               and str(r.get('feature_mode'))==str(ctx['feature_mode'])
               and abs(float(r.get('ratio',-1))-float(ctx['ratio']))<1e-6
               and (pd.isna(r.get('seed')) if seed is None else str(r.get('seed'))==str(seed)) for r in RESULT_ROWS)

def within_budget():
    return (time.monotonic()-JOB_START)/60.0 < MAX_RUNTIME_MINUTES

def record_result(ctx, pred, model, seed, branch, scheme):
    row,frame=evaluate_predictions(ctx,pred,model,seed,branch,scheme)
    RESULT_ROWS.append(row);PREDICTIONS.append(frame)
    pd.DataFrame(RESULT_ROWS).to_csv(OUT/'results_v16.csv',index=False)
    pd.concat(PREDICTIONS,ignore_index=True).to_csv(OUT/'predictions_v16.csv',index=False)
    print(f'[{branch}/{scheme}/{ctx["ratio"]}] {model}, seed={seed}: raw_RMSE={row["RMSE_raw"]:.5f}, MSE_scaled={row["MSE_scaled"]:.6f}')


## 6. Tìm SA trên train/validation — không dùng test


In [ ]:
def fit_validation_only(builder,ctx,hp,epochs):
    r=fit_predict(builder,ctx,hp=hp,epochs=epochs,search=True,skip_test=True)
    return r['val_mse'],r['seconds'],r['epochs_run']


In [ ]:
PAPER_SA = {
    'initial_temperature': 1.0, 'final_temperature': 0.1, 'cooling': 0.95,
    'proposals_per_temperature': 15,  # 650 scenarios / khoảng 45 mức nhiệt
    'proposal_width': 2.0, 'units_min': 8, 'units_max': 128,
    'lr_min': 1e-6, 'lr_max': 0.005,
    'initial': {'units_1': 15, 'units_2': 30, 'learning_rate': 5e-4},
}

def _sigmoid(z):
    return 1.0 / (1.0 + math.exp(-float(np.clip(z, -30, 30))))

def _logit(p):
    p = float(np.clip(p, 1e-6, 1 - 1e-6))
    return math.log(p / (1.0 - p))

def paper_proposal(current, temperature, rng):
    # Bám Eq. 7–8: chuyển tham số sang latent coordinate, perturb theo T,
    # rồi sigmoid trở lại miền hữu hạn. Bounds/proposal_width là giả định cần thiết.
    candidate = {}
    for key in ('units_1', 'units_2', 'learning_rate'):
        if key == 'learning_rate':
            lo, hi, value = math.log(PAPER_SA['lr_min']), math.log(PAPER_SA['lr_max']), math.log(current[key])
        else:
            lo, hi, value = PAPER_SA['units_min'], PAPER_SA['units_max'], current[key]
        latent = _logit((value - lo) / (hi - lo))
        moved = latent + temperature * PAPER_SA['proposal_width'] * rng.uniform(-1.0, 1.0)
        bounded = lo + (hi - lo) * _sigmoid(moved)
        candidate[key] = float(math.exp(bounded)) if key == 'learning_rate' else int(np.clip(round(bounded), lo, hi))
    return candidate

def metropolis(delta, temperature, rng):
    # Paper in exp(+delta/T), nhưng dấu đó cho xác suất > 1 với nghiệm xấu.
    # Dùng dấu âm chuẩn của Metropolis; đây là hiệu chỉnh lỗi công thức được công bố.
    probability = 1.0 if delta <= 0 else math.exp(-delta / max(temperature, 1e-12))
    return bool(rng.random() < probability), probability

def _save_sa_checkpoint(path, state):
    path.write_text(json.dumps(state, indent=2, default=str), encoding='utf-8')

def simulated_annealing_search(search_name, builder, ctx):
    checkpoint = OUT / f'{search_name}_paper_sa_checkpoint.json'
    history_path = OUT / f'{search_name}_paper_sa_history.csv'
    rng = np.random.default_rng(SEED)
    if checkpoint.exists():
        state = json.loads(checkpoint.read_text(encoding='utf-8'))
        rng.bit_generator.state = state['rng_state']
        print(f'Resume {search_name} từ evaluation {state["evaluations"]}')
    else:
        current = dict(PAPER_SA['initial'])
        loss, seconds, epochs_run = fit_validation_only(builder, ctx, current, CFG['search_epochs'])
        state = {
            'current': current, 'current_loss': loss, 'best': dict(current), 'best_loss': loss,
            'evaluations': 1, 'history': [{'evaluation': 1, 'temperature': 1.0,
                'val_mse_scaled': loss, 'best_mse_scaled': loss, 'accepted': True,
                'acceptance_probability': 1.0, **current, 'epochs_run': epochs_run,
                'train_seconds': seconds}],
        }
    while state['evaluations'] < CFG['sa_evals']:
        if not within_budget():
            print('Time budget reached: checkpoint saved; search incomplete')
            break
        proposal_number = state['evaluations'] - 1
        temperature = PAPER_SA['initial_temperature'] * PAPER_SA['cooling'] ** (
            proposal_number // PAPER_SA['proposals_per_temperature'])
        if temperature < PAPER_SA['final_temperature']:
            break
        candidate = paper_proposal(state['current'], temperature, rng)
        loss, seconds, epochs_run = fit_validation_only(builder, ctx, candidate, CFG['search_epochs'])
        accepted, probability = metropolis(loss - state['current_loss'], temperature, rng)
        if accepted:
            state['current'], state['current_loss'] = candidate, loss
        if loss < state['best_loss']:
            state['best'], state['best_loss'] = dict(candidate), loss
        state['evaluations'] += 1
        state['history'].append({'evaluation': state['evaluations'], 'temperature': temperature,
            'val_mse_scaled': loss, 'best_mse_scaled': state['best_loss'], 'accepted': accepted,
            'acceptance_probability': probability, **candidate, 'epochs_run': epochs_run,
            'train_seconds': seconds})
        state['rng_state'] = rng.bit_generator.state
        _save_sa_checkpoint(checkpoint, state)
        pd.DataFrame(state['history']).to_csv(history_path, index=False)
        print(f"{search_name} {state['evaluations']}/{CFG['sa_evals']}: T={temperature:.4f}, val={loss:.6f}, best={state['best_loss']:.6f}")
    state['rng_state'] = rng.bit_generator.state
    _save_sa_checkpoint(checkpoint, state)
    pd.DataFrame(state['history']).to_csv(history_path,index=False)
    result = {'model': search_name, 'best_hp': state['best'],
              'best_val_mse_scaled': state['best_loss'], 'evaluations': state['evaluations'], 'target_evaluations':CFG['sa_evals'], 'search_complete':state['evaluations']>=CFG['sa_evals']}
    (OUT / f'{search_name}_best_hp.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    return state['best'], pd.DataFrame(state['history'])


## 7. Chạy SA lần lượt; dùng checkpoint riêng từng thuật toán


In [ ]:
sa_cache={}
for feature_mode in FEATURE_MODES:
    ctx=prepare_context(data,.90,'expanding',feature_mode)
    tag=f'{feature_mode}_h{HORIZON}_w{LOOKBACK}_{MODE}'
    for name,builder in [('lstm',build_reoptimized_sa_lstm),('gru',build_sa_gru)]:
        hp,hist=simulated_annealing_search(f'sa_{name}_{tag}',builder,ctx)
        sa_cache[name]=hp
(OUT/'sa_best_hyperparameters.json').write_text(json.dumps(sa_cache,indent=2),encoding='utf-8')
print(sa_cache)


## 8. Đánh giá SA mới vs SA công bố (3 splits)

Paper SA-LSTM chỉ là benchmark tham chiếu; SA-GRU không thuộc bài báo. Để tách rõ ảnh hưởng siêu tham số, dùng kiến trúc LSTM chung cho 2 phiên bản.


In [ ]:
PAPER_HP={'units_1':39,'units_2':111,'learning_rate':5e-5}
for ratio in RATIOS:
    ctx=prepare_context(data,ratio,'expanding','paper3')
    jobs=[('paper_sa_lstm',build_matched_sa_lstm,PAPER_HP),
          ('new_sa_lstm',build_matched_sa_lstm,sa_cache['lstm']),
          ('new_sa_gru',build_sa_gru,sa_cache['gru'])]
    for name,builder,hp in jobs:
        if has_result(ctx,name,SEED,'SA_comparison','expanding'):
            print('Skipped finished:',name,ratio);continue
        if not within_budget():
            print('Time budget reached; finished results saved. Re-run to resume.');break
        p=fit_predict(builder,ctx,hp=hp)
        record_result(ctx,p,name,SEED,'SA_comparison','expanding')


## 8B. Chiến lược sau khi tìm được SA hyperparameters

Dùng chính bộ SA-LSTM/SA-GRU đã tìm được và thử **Expanding / Rolling 756 / Recency weighting** trên cùng holdout 90/10; **Residual-SA-GRU** là extension dùng Copper(t−1), không thuộc paper. FULL được giới hạn chỉ **1 seed, 1 split** ở phần chiến lược này để tránh nổ số lượt train. Các kết quả được lưu riêng, không có dependency sang notebook Non-SA.


In [ ]:
STRATEGY_RATIO=.90
for scheme in ['expanding','rolling_756','recency_weighted']:
    ctx=prepare_context(data,STRATEGY_RATIO,scheme,'paper3')
    for name,builder,hp,residual,branch in [
        ('sa_lstm',build_reoptimized_sa_lstm,sa_cache['lstm'],False,'SA_strategy'),
        ('sa_gru',build_sa_gru,sa_cache['gru'],False,'SA_strategy'),
        ('residual_sa_gru',build_sa_gru,sa_cache['gru'],True,'SA_residual_extension')]:
        if has_result(ctx,name,SEED,branch,scheme):
            print('Skipped finished:',name,scheme);continue
        if not within_budget():
            print('Time budget reached; no new SA strategy train started. Re-run to resume.');break
        p=fit_predict(builder,ctx,hp=hp,seed=SEED,scheme=scheme,residual=residual)
        record_result(ctx,p,name,SEED,branch,scheme)


## 9. Báo cáo và đồ thị SA


In [ ]:
if not RESULT_ROWS:
    raise RuntimeError('No results yet; run training cells.')
results=pd.DataFrame(RESULT_ROWS)
predictions=pd.concat(PREDICTIONS,ignore_index=True)
results.to_csv(OUT/'results_v16.csv',index=False)
predictions.to_csv(OUT/'predictions_v16.csv',index=False)
summary=results.groupby(['branch','feature_mode','scheme','model','ratio'],dropna=False)[['MSE_scaled','RMSE_raw','MAE_raw','Relative_MAE_vs_test_persistence']].mean().reset_index()
summary.to_csv(OUT/'summary_v16.csv',index=False)
display(summary.sort_values(['ratio','RMSE_raw']).head(30))

fig,ax=plt.subplots(figsize=(14,5))
subset=predictions[(np.isclose(predictions.ratio,.90)) & (predictions.branch=='SA_comparison')].copy()
for name,g in subset.groupby('model'):
    g=g.sort_values('Date')
    ax.plot(pd.to_datetime(g.Date),g.pred_raw,label=name,alpha=.75)
if not subset.empty:
    g=subset.sort_values('Date').drop_duplicates('Date')
    ax.plot(pd.to_datetime(g.Date),g.actual_raw,label='Copper actual',color='black',linewidth=1.5)
ax.set(title='SA: Actual vs Predicted (90/10)',ylabel='Copper price')
ax.legend();fig.tight_layout();fig.savefig(OUT/'sa_actual_vs_predicted.png',dpi=150);plt.show()


## Xuất ZIP để tải từ Kaggle Output


In [ ]:
summary_lines=[
 '# Copper V16-11 Standalone Results',
 f'- Notebook: {NOTEBOOK_NAME}',
 f'- MODE: {MODE}, HORIZON: {HORIZON}',
 f'- Recorded runs: {len(RESULT_ROWS)}',
 f'- Runtime minutes: {(time.monotonic()-JOB_START)/60:.1f}',
 'FULL_MODE is a budgeted run, not exhaustive paper 650 scenarios.',
]
(OUT/'run_summary.md').write_text('\n'.join(summary_lines),encoding='utf-8')
zip_path=OUT.parent / f'{OUT.name}_results.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in sorted(OUT.rglob('*')):
        if file.is_file():z.write(file,arcname=str(Path(OUT.name)/file.relative_to(OUT)))
print('ZIP:',zip_path,'Size MB:',round(zip_path.stat().st_size/1e6,2))
